In [31]:
!pip install torch torchvision transformers tifffile numpy pandas scikit-learn

In [32]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import numpy as np
import pandas as pd
import tifffile
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import SegformerForSemanticSegmentation

In [62]:
BASE_DIR = "/Users/shaurya/Downloads/PHY:MATH A LVL TOPICALS/India/Mumbai-Slum-detection-and-prediction/data"

tiles = pd.read_csv(os.path.join(BASE_DIR, "tiles", "tiles_out", "tile_index.csv"))
tiles["tiles_dir"] = os.path.join(BASE_DIR, "tiles", "tiles_out")

max_col = tiles["col"].max()
val_cols = set(range(max_col - 1, max_col + 1))

train_df = tiles[~tiles["col"].isin(val_cols)].reset_index(drop=True)
val_df = tiles[tiles["col"].isin(val_cols)].reset_index(drop=True)

print(f"Train tiles: {len(train_df)}, Val tiles: {len(val_df)}")
print(f"Train slum tiles: {(train_df.slum_fraction > 0).sum()}, "
      f"Val slum tiles: {(val_df.slum_fraction > 0).sum()}")

Train tiles: 75, Val tiles: 30
Train slum tiles: 56, Val slum tiles: 13


In [63]:
class SlumDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df = df
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(row["tiles_dir"], row["filename"])

        arr = tifffile.imread(path)
        image = arr[:10].astype(np.float32)
        mask = arr[10].astype(np.int64)

        image = np.clip(image / 10000.0, 0, 1)
        image = np.nan_to_num(image, nan=0.0)

        image = torch.from_numpy(image)
        mask = torch.from_numpy(mask)
        return image, mask


train_ds = SlumDataset(train_df, augment=False)
val_ds = SlumDataset(val_df, augment=False)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False)

img, msk = train_ds[0]
print(img.shape, msk.shape, msk.unique())

torch.Size([10, 256, 256]) torch.Size([256, 256]) tensor([0])


In [64]:
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b0",
    num_labels=2,
    ignore_mismatched_sizes=True,
)

old_conv = model.segformer.stages[0].patch_embeddings.proj
new_conv = torch.nn.Conv2d(
    in_channels=10,
    out_channels=old_conv.out_channels,
    kernel_size=old_conv.kernel_size,
    stride=old_conv.stride,
    padding=old_conv.padding,
)

with torch.no_grad():
    new_conv.weight[:, :3] = old_conv.weight
    torch.nn.init.kaiming_normal_(new_conv.weight[:, 3:])
    new_conv.bias.copy_(old_conv.bias)

model.segformer.stages[0].patch_embeddings.proj = new_conv

device = torch.device("cpu")
model.to(device)
print(f"Using device: {device}")

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.weight                                       | UNEXPECTED | 
classifier.bias                                         | UNEXPECTED | 
decode_head.batch_norm.running_mean                     | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.weight                           | MISSING    | 
decode_head.batch_norm.num_batches_tracked              | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.batch_norm.running_var                      | MISSING    | 
decode_head.batch_norm.bias                             | MISSING    | 
decode_head.classifier.weight                           | MISSING    | 
decode_head.linear_fuse.weight                          

Using device: cpu


In [65]:
import torch.nn as nn
import torch.nn.functional as F

class_weights = torch.tensor([1.0, 5.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

In [67]:
optimizer = torch.optim.AdamW(model.parameters(), lr=6e-5)

In [68]:
num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(pixel_values=images)
        logits = F.interpolate(outputs.logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader)

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(pixel_values=images)
            logits = F.interpolate(outputs.logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
            loss = criterion(logits, masks)
            val_loss += loss.item()
    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch {epoch+1}/{num_epochs} - train loss: {avg_train_loss:.4f} - val loss: {avg_val_loss:.4f}")

Epoch 1/20 - train loss: 0.6163 - val loss: 0.6499
Epoch 2/20 - train loss: 0.5102 - val loss: 0.5232
Epoch 3/20 - train loss: 0.4552 - val loss: 0.5799
Epoch 4/20 - train loss: 0.4025 - val loss: 0.3057
Epoch 5/20 - train loss: 0.3621 - val loss: 0.3257
Epoch 6/20 - train loss: 0.3324 - val loss: 0.2405
Epoch 7/20 - train loss: 0.3227 - val loss: 0.2556
Epoch 8/20 - train loss: 0.3031 - val loss: 0.2502
Epoch 9/20 - train loss: 0.2905 - val loss: 0.2579
Epoch 10/20 - train loss: 0.2965 - val loss: 0.1977
Epoch 11/20 - train loss: 0.2742 - val loss: 0.1778
Epoch 12/20 - train loss: 0.2683 - val loss: 0.1969
Epoch 13/20 - train loss: 0.2825 - val loss: 0.1750
Epoch 14/20 - train loss: 0.2553 - val loss: 0.1876
Epoch 15/20 - train loss: 0.2473 - val loss: 0.1727
Epoch 16/20 - train loss: 0.2507 - val loss: 0.1746
Epoch 17/20 - train loss: 0.2541 - val loss: 0.1640
Epoch 18/20 - train loss: 0.2494 - val loss: 0.1516
Epoch 19/20 - train loss: 0.2351 - val loss: 0.1456
Epoch 20/20 - train l

In [69]:
model.eval()
total_correct, total_pixels = 0, 0
intersection, union = 0, 0
true_positive, false_positive, false_negative = 0, 0, 0

with torch.no_grad():
    for images, masks in val_loader:
        images, masks = images.to(device), masks.to(device)
        outputs = model(pixel_values=images)
        logits = F.interpolate(outputs.logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
        preds = torch.argmax(logits, dim=1)

        total_correct += (preds == masks).sum().item()
        total_pixels += masks.numel()
        pred_slum = (preds == 1)
        true_slum = (masks == 1)
        intersection += (pred_slum & true_slum).sum().item()
        union += (pred_slum | true_slum).sum().item()
        true_positive += (pred_slum & true_slum).sum().item()
        false_positive += (pred_slum & ~true_slum).sum().item()
        false_negative += (~pred_slum & true_slum).sum().item()

pixel_accuracy = total_correct / total_pixels
iou_slum = intersection / union if union > 0 else 0
precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) > 0 else 0
recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Pixel accuracy: {pixel_accuracy:.4f}")
print(f"Slum IoU: {iou_slum:.4f}")
print(f"Slum precision: {precision:.4f}")
print(f"Slum recall: {recall:.4f}")
print(f"Slum F1: {f1:.4f}")

Pixel accuracy: 0.9675
Slum IoU: 0.4953
Slum precision: 0.5872
Slum recall: 0.7600
Slum F1: 0.6625


In [61]:
model.eval()
best_f1, best_threshold = 0, 0.5

for threshold in np.arange(0.3, 0.98, 0.03):
    tp, fp, fn = 0, 0, 0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(pixel_values=images)
            logits = F.interpolate(outputs.logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)
            probs = torch.softmax(logits, dim=1)[:, 1]  # probability of "slum" class
            preds = (probs > threshold)
            true = (masks == 1)
            tp += (preds & true).sum().item()
            fp += (preds & ~true).sum().item()
            fn += (~preds & true).sum().item()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    print(f"threshold={threshold:.2f}  precision={precision:.3f}  recall={recall:.3f}  f1={f1:.3f}")
    if f1 > best_f1:
        best_f1, best_threshold = f1, threshold

print(f"\nBest threshold: {best_threshold:.2f}, F1: {best_f1:.4f}")

threshold=0.30  precision=0.457  recall=0.844  f1=0.593
threshold=0.33  precision=0.476  recall=0.827  f1=0.605
threshold=0.36  precision=0.494  recall=0.810  f1=0.614
threshold=0.39  precision=0.511  recall=0.792  f1=0.621
threshold=0.42  precision=0.528  recall=0.775  f1=0.628
threshold=0.45  precision=0.544  recall=0.755  f1=0.632
threshold=0.48  precision=0.560  recall=0.734  f1=0.635
threshold=0.51  precision=0.576  recall=0.713  f1=0.637
threshold=0.54  precision=0.592  recall=0.693  f1=0.638
threshold=0.57  precision=0.608  recall=0.669  f1=0.637
threshold=0.60  precision=0.622  recall=0.645  f1=0.634
threshold=0.63  precision=0.635  recall=0.621  f1=0.628
threshold=0.66  precision=0.648  recall=0.596  f1=0.621
threshold=0.69  precision=0.661  recall=0.569  f1=0.612
threshold=0.72  precision=0.674  recall=0.542  f1=0.601
threshold=0.75  precision=0.688  recall=0.511  f1=0.587
threshold=0.78  precision=0.702  recall=0.477  f1=0.568
threshold=0.81  precision=0.716  recall=0.439  f

In [11]:
import os
import pandas as pd
import glob

BASE_DIR = "/Users/shaurya/Downloads/PHY:MATH A LVL TOPICALS/India/Mumbai-Slum-detection-and-prediction/data"

table_paths = glob.glob(os.path.join(BASE_DIR, "grids", "table_*.csv"))
all_tables = pd.concat([pd.read_csv(p) for p in table_paths], ignore_index=True)

print(f"Total rows: {len(all_tables)}")
print(f"Expanded: {all_tables['expanded'].sum()}, Not expanded: {(all_tables['expanded']==0).sum()}")
print(f"Flagged unexpected shrinks (review): {all_tables['unexpected_shrink'].sum()}")

Total rows: 161200
Expanded: 1855, Not expanded: 159345
Flagged unexpected shrinks (review): 1171


In [14]:
# Exclude cells that were already near the 50% threshold before the
# transition - these are the ones most likely to "expand" purely from
# small year-to-year prediction noise, not real growth.
filtered = all_tables[
    (all_tables["slum_fraction_before"] < 0.35) | (all_tables["was_slum"] == 1)
].copy()

# Keep only genuinely clear-cut cases: cells that were clearly NOT slum
# before (comfortably below threshold), so "expanded" means a real jump.
filtered = filtered[all_tables["slum_fraction_before"] < 0.35]

print(f"Rows before filtering: {len(all_tables)}, after: {len(filtered)}")
print(f"Expanded: {filtered['expanded'].sum()}, Not expanded: {(filtered['expanded']==0).sum()}")

feature_cols = [c for c in filtered.columns if c.startswith("dist_to_")] + \
               ["slum_fraction_before", "neighbor_slum_fraction"]
feature_cols = [c for c in feature_cols if c in filtered.columns]

X = filtered[feature_cols]
y = filtered["expanded"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf2 = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight="balanced", random_state=42)
rf2.fit(X_train, y_train)

print(classification_report(y_test, rf2.predict(X_test)))

importances2 = pd.Series(rf2.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances2)

/var/folders/n9/1xc7bk4s09l6m8fxgkkmjk0r0000gn/T/ipykernel_84901/945889443.py:10: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  filtered = filtered[all_tables["slum_fraction_before"] < 0.35]


Rows before filtering: 161200, after: 152016
Expanded: 1057, Not expanded: 150959
              precision    recall  f1-score   support

           0       1.00      0.86      0.92     30193
           1       0.03      0.60      0.06       211

    accuracy                           0.86     30404
   macro avg       0.51      0.73      0.49     30404
weighted avg       0.99      0.86      0.92     30404

slum_fraction_before      0.454992
dist_to_nearest_slum_m    0.417474
neighbor_slum_fraction    0.127534
dist_to_railway_m         0.000000
dist_to_industry_m        0.000000
dist_to_road_m            0.000000
dist_to_water_m           0.000000
dtype: float64


In [16]:
feature_cols = ["slum_fraction_before", "neighbor_slum_fraction", "dist_to_nearest_slum_m"]
feature_cols = [c for c in feature_cols if c in all_tables.columns]
print("Using features:", feature_cols)

X = all_tables[feature_cols]
y = all_tables["expanded"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf_final = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight="balanced", random_state=42)
rf_final.fit(X_train, y_train)

print(classification_report(y_test, rf_final.predict(X_test)))

importances_final = pd.Series(rf_final.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances_final)

Using features: ['slum_fraction_before', 'neighbor_slum_fraction', 'dist_to_nearest_slum_m']
              precision    recall  f1-score   support

           0       1.00      0.92      0.96     31869
           1       0.10      0.75      0.18       371

    accuracy                           0.92     32240
   macro avg       0.55      0.83      0.57     32240
weighted avg       0.99      0.92      0.95     32240

slum_fraction_before      0.496500
dist_to_nearest_slum_m    0.325726
neighbor_slum_fraction    0.177774
dtype: float64


In [19]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

probs = rf_final.predict_proba(X_test)[:, 1]  # probability of "expanded"

print(f"{'Threshold':<10}{'Precision':<12}{'Recall':<10}{'F1':<10}")
best_f1, best_threshold = 0, 0.5
for t in np.arange(0.85, 0.995, 0.01):
    preds = (probs > t).astype(int)
    p = precision_score(y_test, preds, zero_division=0)
    r = recall_score(y_test, preds, zero_division=0)
    f1 = f1_score(y_test, preds, zero_division=0)
    print(f"{t:<10.2f}{p:<12.3f}{r:<10.3f}{f1:<10.3f}")
    if f1 > best_f1:
        best_f1, best_threshold = f1, t

print(f"\nBest threshold: {best_threshold:.2f}, F1: {best_f1:.4f}")

Threshold Precision   Recall    F1        
0.85      0.221       0.633     0.328     
0.86      0.229       0.623     0.335     
0.87      0.231       0.604     0.334     
0.88      0.247       0.574     0.345     
0.89      0.252       0.563     0.348     
0.90      0.255       0.558     0.350     
0.91      0.263       0.555     0.357     
0.92      0.271       0.531     0.359     
0.93      0.283       0.499     0.361     
0.94      0.298       0.466     0.363     
0.95      0.316       0.445     0.370     
0.96      0.352       0.394     0.372     
0.97      0.427       0.275     0.334     
0.98      0.571       0.022     0.042     
0.99      0.667       0.005     0.011     

Best threshold: 0.96, F1: 0.3715


In [13]:
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances)

slum_fraction_before      0.530357
dist_to_nearest_slum_m    0.342956
neighbor_slum_fraction    0.126687
dist_to_railway_m         0.000000
dist_to_industry_m        0.000000
dist_to_road_m            0.000000
dist_to_water_m           0.000000
dtype: float64


In [10]:
import os
import pandas as pd
import glob

BASE_DIR = "/Users/shaurya/Downloads/PHY:MATH A LVL TOPICALS/India/Mumbai-Slum-detection-and-prediction/data"

table_paths = glob.glob(os.path.join(BASE_DIR, "grids", "table_*.csv"))
print("Files found:", table_paths)

all_tables = pd.concat([pd.read_csv(p) for p in table_paths], ignore_index=True)
print("Columns:", all_tables.columns.tolist())
print("Total rows:", len(all_tables))

Files found: ['/Users/shaurya/Downloads/PHY:MATH A LVL TOPICALS/India/Mumbai-Slum-detection-and-prediction/data/grids/table_2017_2018.csv', '/Users/shaurya/Downloads/PHY:MATH A LVL TOPICALS/India/Mumbai-Slum-detection-and-prediction/data/grids/table_2019_2020.csv', '/Users/shaurya/Downloads/PHY:MATH A LVL TOPICALS/India/Mumbai-Slum-detection-and-prediction/data/grids/table_2016_2017.csv', '/Users/shaurya/Downloads/PHY:MATH A LVL TOPICALS/India/Mumbai-Slum-detection-and-prediction/data/grids/table_2018_2019.csv', '/Users/shaurya/Downloads/PHY:MATH A LVL TOPICALS/India/Mumbai-Slum-detection-and-prediction/data/grids/table_2020_2021.csv']
Columns: ['grid_row', 'grid_col', 'slum_fraction_before', 'was_slum', 'neighbor_slum_fraction', 'dist_to_nearest_slum_m', 'slum_fraction_after', 'is_slum_after', 'dist_to_railway_m', 'dist_to_industry_m', 'dist_to_road_m', 'dist_to_water_m', 'expanded', 'unexpected_shrink']
Total rows: 161200


In [15]:
print(all_tables[["dist_to_railway_m", "dist_to_industry_m", "dist_to_road_m", "dist_to_water_m"]].describe())

       dist_to_railway_m  dist_to_industry_m  dist_to_road_m  dist_to_water_m
count                0.0                 0.0             0.0              0.0
mean                 NaN                 NaN             NaN              NaN
std                  NaN                 NaN             NaN              NaN
min                  NaN                 NaN             NaN              NaN
25%                  NaN                 NaN             NaN              NaN
50%                  NaN                 NaN             NaN              NaN
75%                  NaN                 NaN             NaN              NaN
max                  NaN                 NaN             NaN              NaN


In [20]:
import joblib
import os

MODELS_DIR = os.path.join(BASE_DIR, "..", "models")
os.makedirs(MODELS_DIR, exist_ok=True)

FINAL_THRESHOLD = 0.75  # your chosen operating threshold from the tuning discussion

joblib.dump({"model": rf_final, "threshold": FINAL_THRESHOLD}, os.path.join(MODELS_DIR, "random_forest_expansion.pkl"))
print(f"Saved model + threshold to {MODELS_DIR}")

Saved model + threshold to /Users/shaurya/Downloads/PHY:MATH A LVL TOPICALS/India/Mumbai-Slum-detection-and-prediction/data/../models
